# 🚀 Soro-Kev: System-One Fast Tool Routing & Argument Repair
### End-to-End GPU Training & Split Conformal Evaluation (Kaggle 2x T4 / Colab / A6000)

**Soro-Kev** is a calibrated System-One pointer decision model (based on `jaredpalmer/kev` and `Qwen2.5/3`) that executes in under **15ms**:
1. **Tool Routing:** Ranks 4–10 candidate tools across 5 polymorphic developer formats (`mcp_json`, `agent_skill`, `rest_openapi`, `python_sig`, `graphql_rpc`).
2. **Permutation Invariance:** Evaluates all candidates in **ONE** single-sequence pass with cross-option causal attention and enforces $\mathcal{L}_{\text{JSD}}(p \parallel p_{\text{perm}}) \to 0$.
3. **Brier Proper Scoring & Split Conformal Calibration:** Quantifies uncertainty with mathematical coverage guarantees (target: $1 - \alpha = 90\%$) and shrinks candidate prediction sets down to singletons ($|S|=1$) for autonomous `EXECUTE`.
4. **Auxiliary Parameter Repair:** Solves semantic argument mapping, missing required argument detection, and state mutation risk classification.

## 1. Inspect GPU Acceleration Environment

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
    print(f"BF16 Supported:  {torch.cuda.is_bf16_supported()}")

## 2. Install Required Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers peft accelerate pyarrow numpy

## 3. Clone Repository & Setup Environment

In [ ]:
## 4. Synthesize Universal Polymorphic Dataset & Real Human Benchmarks
Generates 37,000+ multi-turn polymorphic records partitioned into:
- **Train Set (34,000+ records):** 300+ training servers blended with 1,300+ real human benchmark queries.
- **Zero-Shot Validation Set (3,000 records):** 50+ held-out servers.
- **External Human Benchmark (600 records):** Pristine real non-template queries from mcp-routing, tau2, MCPAgentBench.
- Strips verbatim docstring sentences to prevent lexical shortcut leakage.

!python training/build_universal_dataset.py \
    --catalog data/mcp_catalog/tools.parquet \
    --output-dir data/soro_kev_dataset \
    --train-count 34000 \
    --val-count 3000 \
    --val-external-count 600 \
    --train-ratio 0.85

print("\n=== AUDITING DATASET INVARIANTS ===")
!python training/audit_dataset.py --data-dir data/soro_kev_dataset

In [ ]:
## 5. Train Soro-Kev on GPU with Sequence Packing & Mini-Batching
Runs dual pointer heads + LoRA adapter with:
- Multi-question sequence packing cutting 5 forward passes down to 2
- True GPU mini-batching with right-padded attention masks
- Brier proper scoring penalty $\lambda_{\text{Brier}} = 0.25$
- Permutation JSD loss $\lambda_{\text{JSD}} = 0.30$
- Cosine learning rate schedule with linear warmup
- Periodic split conformal validation saving the best checkpoint

*Tip: Switch `--base-model` to `Qwen/Qwen3.5-0.8B-Base` or `Qwen/Qwen3.5-4B-Base` as GPU VRAM allows.*

!python training/train_soro_kev.py \
    --profile kaggle_t4 \
    --base-model Qwen/Qwen3-0.6B \
    --batch-size 4 \
    --data-dir data/soro_kev_dataset \
    --output-dir runs/soro_kev_run \
    --epochs 3 \
    --lr 2e-4 \
    --grad-accum 4 \
    --eval-every 500

In [ ]:
## 6. Split Conformal Benchmark Evaluation
Evaluates on **both** zero-shot held-out packages AND the real human external benchmark:
1. `val_zeroshot.jsonl` (3,000 records): Tests generalization to completely unseen MCP tool servers.
2. `val_external_human.jsonl` (600 records): Tests robustness on genuine real-world developer queries.

print("=== ARM 1: HEURISTIC BASELINE (ZERO-SHOT VAL) ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_zeroshot.jsonl \
    --output-report reports/benchmark_heuristic.json \
    --backend heuristic \
    --alpha 0.10 \
    --cal-size 300

print("\n=== ARM 2: TRAINED NEURAL SORO-KEV MODEL (ZERO-SHOT VAL) ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_zeroshot.jsonl \
    --output-report reports/benchmark_neural_soro_kev.json \
    --backend neural \
    --checkpoint runs/soro_kev_run \
    --alpha 0.10 \
    --cal-size 300

print("\n=== ARM 3: HEURISTIC BASELINE (EXTERNAL HUMAN BENCHMARK) ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_external_human.jsonl \
    --output-report reports/benchmark_human_heuristic.json \
    --backend heuristic \
    --alpha 0.10 \
    --cal-size 150

print("\n=== ARM 4: TRAINED NEURAL SORO-KEV MODEL (EXTERNAL HUMAN BENCHMARK) ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_external_human.jsonl \
    --output-report reports/benchmark_human_neural.json \
    --backend neural \
    --checkpoint runs/soro_kev_run \
    --alpha 0.10 \
    --cal-size 150

In [ ]:
print("=== ARM 1: HEURISTIC BASELINE ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_zeroshot.jsonl \
    --output-report reports/benchmark_heuristic.json \
    --backend heuristic \
    --alpha 0.10 \
    --cal-size 300

print("\n=== ARM 2: TRAINED NEURAL SORO-KEV MODEL ===")
!python training/evaluate_benchmark.py \
    --val-data data/soro_kev_dataset/val_zeroshot.jsonl \
    --output-report reports/benchmark_neural_soro_kev.json \
    --backend neural \
    --checkpoint runs/soro_kev_run \
    --alpha 0.10 \
    --cal-size 300

## 7. Comparative Performance Scorecard (Heuristic vs. Trained Soro-Kev)

In [ ]:
import json
from pathlib import Path

rep_heu = json.loads(Path("reports/benchmark_heuristic.json").read_text(encoding="utf-8"))
rep_neu = json.loads(Path("reports/benchmark_neural_soro_kev.json").read_text(encoding="utf-8"))

print("=" * 85)
print(f"{'Metric (900 Zero-Shot Held-Out Records)':<45} | {'Heuristic Baseline':<16} | {'Trained Soro-Kev':<16}")
print("=" * 85)
print(f"{'Top-1 Zero-Shot Route Accuracy':<45} | {rep_heu['overall_route_accuracy']*100:>14.2f}% | {rep_neu['overall_route_accuracy']*100:>14.2f}%")
print(f"{'Permutation Flip Rate (<5% target)':<45} | {rep_heu['permutation_flip_rate']*100:>14.2f}% | {rep_neu['permutation_flip_rate']*100:>14.2f}%")
print(f"{'Conformal Empirical Coverage (target: 90%)':<45} | {rep_heu['conformal_calibration']['empirical_coverage']*100:>14.2f}% | {rep_neu['conformal_calibration']['empirical_coverage']*100:>14.2f}%")
print(f"{'Average Conformal Set Size |S|':<45} | {rep_heu['conformal_calibration']['mean_set_size']:>14.2f}  | {rep_neu['conformal_calibration']['mean_set_size']:>14.2f} ")
print(f"{'Autonomous EXECUTE Rate (|S|=1)':<45} | {rep_heu['conformal_calibration']['singleton_rate']*100:>14.2f}% | {rep_neu['conformal_calibration']['singleton_rate']*100:>14.2f}%")
print(f"{'Disambiguation ASK Rate (2 <= |S| <= 8)':<45} | {rep_heu['conformal_calibration']['action_distribution']['ASK_share']*100:>14.2f}% | {rep_neu['conformal_calibration']['action_distribution']['ASK_share']*100:>14.2f}%")
print(f"{'Semantic Arg Mapping Accuracy':<45} | {rep_heu['auxiliary_tasks']['arg_mapping_accuracy']*100:>14.2f}% | {rep_neu['auxiliary_tasks']['arg_mapping_accuracy']*100:>14.2f}%")
print(f"{'Missing Required Arg Detection':<45} | {rep_heu['auxiliary_tasks']['missing_required_detection_accuracy']*100:>14.2f}% | {rep_neu['auxiliary_tasks']['missing_required_detection_accuracy']*100:>14.2f}%")
print(f"{'Mutation Risk Classification':<45} | {rep_heu['auxiliary_tasks']['is_mutation_detection_accuracy']*100:>14.2f}% | {rep_neu['auxiliary_tasks']['is_mutation_detection_accuracy']*100:>14.2f}%")
print("=" * 85)

## 8. Package Trained Checkpoint for Download & Hugging Face Upload

In [ ]:
!zip -r runs/soro_kev_checkpoint.zip runs/soro_kev_run

from IPython.display import FileLink
print("Click link below to download trained Soro-Kev weights:")
display(FileLink("runs/soro_kev_checkpoint.zip"))